<a href="https://colab.research.google.com/github/RaihahMahmud/FlyRank-AI--starter-ML-Internship-/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


I use Logistic Regression as the first modeling method.

The goal of this lane is to rank content pages for review based on observed search signals. Logistic Regression is appropriate as a simple, interpretable baseline model because it can estimate the likelihood that a page matches the CTR opportunity pattern.

I will use search visibility and engagement-related signals available in the March 2026 data. I will avoid future-window fields and fields derived from future outcomes.

The model is intended for decision-support and ranking, not to claim that refreshing a page will cause better Google performance.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I use a train/test split on the same March 2026 page-level dataset used for the baseline comparison.

The split is performed before model fitting so that the test data is not used to train the model.

The baseline and model are evaluated on the same held-out test pages using the same ranking-oriented metric.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I trained a Random Forest model using the same train/test split and evaluation metric as my Week-4 baseline. The baseline provides a simple reference point based on the training data.

The comparison shows whether the model provides a measurable improvement over that simple reference. I treat this as decision-support evidence rather than proof that the model will improve page performance.

In [26]:
# Reliable dataset setup

import os
import duckdb
import pandas as pd
import numpy as np

from huggingface_hub import login, hf_hub_download
from google.colab import userdata

# 1. Hugging Face authentication


HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN is missing from Colab Secrets.")

login(token=HF_TOKEN, add_to_git_credential=False)

print("Hugging Face login: successful")



local_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=HF_TOKEN
)

print("Dataset downloaded successfully:")
print(local_file)


# 3. DuckDB


con = duckdb.connect()

print("DuckDB version:", duckdb.__version__)

# 4. Test local Parquet

test = con.sql(f"""
    SELECT COUNT(*) AS total_rows
    FROM read_parquet('{local_file}')
""").df()

print("March 2026 data access test:")
display(test)

Hugging Face login: successful
Dataset downloaded successfully:
/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet
DuckDB version: 1.3.2
March 2026 data access test:


,total_rows
0,9841378


In [27]:
# 3. TRAIN + COMPARE VS BASELINE


import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# 1. Use the already-loaded FULL March dataframe

print("Using already-loaded March data...")
print("March rows:", len(march))
print("March dates:", march["report_date"].nunique())

# Keep only columns needed for this model
march_model = march[
    [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position"
    ]
].copy()

march_model = march_model.dropna(
    subset=["gsc_impressions", "gsc_clicks"]
)

# 2. Aggregate March → one row per content page/client

march_model["report_date"] = pd.to_datetime(
    march_model["report_date"]
)

march_model["ctr"] = (
    march_model["gsc_clicks"] /
    march_model["gsc_impressions"].replace(0, np.nan)
).fillna(0)

base = (
    march_model
    .groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        gsc_impressions=("gsc_impressions", "sum"),
        gsc_clicks=("gsc_clicks", "sum"),
        ctr=("ctr", "mean"),
        gsc_avg_position=("gsc_avg_position", "mean")
    )
)

print("Unique content/client rows:", len(base))

# 3. Create Week-4 baseline score

base["priority_score"] = (
    (base["gsc_impressions"] >
     base["gsc_impressions"].quantile(0.75))
    &
    (base["ctr"] <
     base["ctr"].quantile(0.25))
).astype(int)

base["baseline_prediction"] = base["priority_score"]

print("\nBaseline distribution:")
print(base["baseline_prediction"].value_counts())

# ------------------------------------------------------------
# 4. Create modeling target
# ------------------------------------------------------------

click_threshold = base["gsc_clicks"].median()

base["target"] = (
    base["gsc_clicks"] > click_threshold
).astype(int)

print("\nTarget distribution:")
print(base["target"].value_counts())


feature_cols = [
    "gsc_impressions",
    "ctr",
    "gsc_avg_position"
]

model_df = base.dropna(
    subset=feature_cols + ["target", "client_hash_id"]
).copy()

X = model_df[feature_cols]
y = model_df["target"]
groups = model_df["client_hash_id"]

print("\nRows used for modeling:", len(model_df))
print("Unique clients:", groups.nunique())

# 6. Grouped train/test split


splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("\nTrain rows:", len(X_train))
print("Test rows:", len(X_test))

# 7. Train Logistic Regression

model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

# 8. Predictions

model_probability = model.predict_proba(X_test)[:, 1]

model_prediction = (
    model_probability >= 0.5
).astype(int)

# 9. Baseline predictions on SAME test set

baseline_prediction = (
    model_df["baseline_prediction"]
    .iloc[test_idx]
    .to_numpy()
)

# 10. Metrics

def calculate_metrics(
    y_true,
    prediction,
    probability=None
):

    results = {
        "Precision": precision_score(
            y_true,
            prediction,
            zero_division=0
        ),
        "Recall": recall_score(
            y_true,
            prediction,
            zero_division=0
        ),
        "F1": f1_score(
            y_true,
            prediction,
            zero_division=0
        )
    }

    if probability is not None:

        results["Average Precision"] = (
            average_precision_score(
                y_true,
                probability
            )
        )

        if len(np.unique(y_true)) == 2:

            results["ROC-AUC"] = roc_auc_score(
                y_true,
                probability
            )

    return results


baseline_metrics = calculate_metrics(
    y_test,
    baseline_prediction
)

model_metrics = calculate_metrics(
    y_test,
    model_prediction,
    model_probability
)

comparison = pd.DataFrame([
    {
        "Method": "Week-4 baseline",
        **baseline_metrics
    },
    {
        "Method": "Logistic Regression",
        **model_metrics
    }
])

print("\n================ MODEL VS BASELINE ================\n")
display(comparison.round(4))

# 11. Feature interpretation

lr = model.named_steps["model"]

feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": lr.coef_[0],
    "absolute_coefficient": np.abs(lr.coef_[0])
}).sort_values(
    "absolute_coefficient",
    ascending=False
)

print("\nFeature interpretation:")
display(feature_importance.round(4))

# 12. Prepare results for Section 4

test_results = model_df.iloc[test_idx].copy()

test_results["actual"] = y_test.to_numpy()
test_results["prediction"] = model_prediction
test_results["probability"] = model_probability

test_results["error"] = (
    test_results["actual"] !=
    test_results["prediction"]
)

print("\nModeling complete.")
print("Model: Logistic Regression")
print("Baseline: Week-4 CTR opportunity rule")
print("Validation: Grouped by client")
print("Comparison uses the SAME test rows and SAME target.")

Using already-loaded March data...
March rows: 9841378
March dates: 31
Unique content/client rows: 331437

Baseline distribution:
baseline_prediction
0    331437
Name: count, dtype: int64

Target distribution:
target
0    262600
1     68837
Name: count, dtype: int64

Rows used for modeling: 176738
Unique clients: 47

Train rows: 138310
Test rows: 38428

================ MODEL VS BASELINE ================



,Method,Precision,Recall,F1,Average Precision,ROC-AUC
0,Week-4 baseline,0.0000,0.0000,0.0000,NaN,NaN
1,Logistic Regression,0.9993,0.9863,0.9927,0.9999,0.9999



Feature interpretation:


,feature,coefficient,absolute_coefficient
1,ctr,57.6148,57.6148
0,gsc_impressions,4.4000,4.4000
2,gsc_avg_position,-0.2854,0.2854



Modeling complete.
Model: Logistic Regression
Baseline: Week-4 CTR opportunity rule
Validation: Grouped by client
Comparison uses the SAME test rows and SAME target.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*
==>

The model made 255 errors out of 38,428 test observations, giving an observed error rate of 0.66%. The errors were concentrated among pages with very low CTR and low click counts.

The model relied most on CTR, followed by impressions and average position. Higher CTR and impressions were associated with a higher probability of the positive class, while lower average position was associated with a higher probability.

These results are directional and should be treated as decision-support evidence. The target is based on observed click volume, so the strong measured performance should not be interpreted as evidence that the model predicts the effect of refreshing a page.

In [28]:
# 4. ERRORS AND INTERPRETATION


error_count = test_results["error"].sum()
total_test = len(test_results)
error_rate = error_count / total_test

print("Error analysis")
print("----------------")
print(f"Test rows: {total_test:,}")
print(f"Incorrect predictions: {error_count:,}")
print(f"Error rate: {error_rate:.4f}")

# 2. Correct vs incorrect predictions


error_summary = (
    test_results
    .groupby("error")[
        [
            "gsc_impressions",
            "gsc_clicks",
            "ctr",
            "gsc_avg_position"
        ]
    ]
    .mean()
    .rename(index={
        False: "Correct",
        True: "Incorrect"
    })
)

print("\nAverage characteristics:")
display(error_summary.round(4))

# 3. Example errors

print("\nExample model errors:")

display(
    test_results[
        test_results["error"]
    ][[
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "gsc_avg_position",
        "actual",
        "prediction",
        "probability"
    ]]
    .sort_values("probability")
    .head(10)
)

# 4. What the model leaned on

print("\nFeature interpretation:")
display(
    feature_importance[
        [
            "feature",
            "coefficient",
            "absolute_coefficient"
        ]
    ].round(4)
)

# 5. Simple interpretation

print("\nInterpretation:")

for _, row in feature_importance.iterrows():

    direction = (
        "higher"
        if row["coefficient"] > 0
        else "lower"
    )

    print(
        f"- {row['feature']}: {direction} values "
        f"are associated with a higher predicted probability "
        f"of the positive class."
    )

Error analysis
----------------
Test rows: 38,428
Incorrect predictions: 255
Error rate: 0.0066

Average characteristics:


,gsc_impressions,gsc_clicks,ctr,gsc_avg_position
error,,,,
Correct,2056.9141,6.5436,0.0021,16.0635
Incorrect,1964.0353,1.0588,0.0002,8.7410



Example model errors:


,content_hash_id,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,actual,prediction,probability
215537,content_0cc0d0db31ad44d1,883,1,0.000041,3.122445,1,0,0.027762
241635,content_efcb6aec8e61566f,370,1,0.000097,5.027013,1,0,0.030622
220185,content_349f8cc076850c31,1128,1,0.000046,8.981635,1,0,0.032621
239149,content_d9fd8d0dbbc504ce,370,1,0.000103,0.175000,1,0,0.034800
231452,content_979ed4a9dffedf31,324,1,0.000109,1.853968,1,0,0.034827
30829,content_fde854d58f98fff2,599,1,0.000092,5.301767,1,0,0.034846
220940,content_3b0603528ba7e3a7,1511,1,0.000036,9.090721,1,0,0.040268
29769,content_667f5dfc446b591c,1336,1,0.000065,2.619213,1,0,0.051172
97154,content_12cf780af24cf6cf,787,1,0.000188,46.680097,1,0,0.052736
216049,content_10f6fcfd83eb1e9c,1258,1,0.000099,6.605714,1,0,0.061697



Feature interpretation:


,feature,coefficient,absolute_coefficient
1,ctr,57.6148,57.6148
0,gsc_impressions,4.4000,4.4000
2,gsc_avg_position,-0.2854,0.2854



Interpretation:
- ctr: higher values are associated with a higher predicted probability of the positive class.
- gsc_impressions: higher values are associated with a higher predicted probability of the positive class.
- gsc_avg_position: lower values are associated with a higher predicted probability of the positive class.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.